# Data Visualization

All figures for the conference paper — dataset overview, physical signals, feature space, and final-model results.

All figures are saved to `figures/` with the prefix `viz_*`. Re-run any cell to regenerate.

Sections:
1. Dataset overview (counts, durations)
2. Raw signals — waveforms (good vs bad)
3. Spectral evidence — mel spectrograms (good vs bad + difference map)
4. Feature space PCA (training-only, then with holdout overlay)
5. Final-model results: per-pillar P(bad), per-clip predictions, confusion matrix, before/after
6. Pipeline diagram

In [1]:
import os, glob, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
import librosa, librosa.display
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix
warnings.filterwarnings('ignore')

# Project root — auto-detected by walking up until data/ and model/ are found.
ROOT = os.getcwd()
while not (os.path.isdir(os.path.join(ROOT, 'data')) and os.path.isdir(os.path.join(ROOT, 'model'))):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError('Project root not found (no data/ and model/ above current directory)')
    ROOT = parent
FIG = os.path.join(ROOT, 'figures')
os.makedirs(FIG, exist_ok=True)

SR = 16000
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'sans-serif',
                     'axes.spines.top': False, 'axes.spines.right': False})
print(f'Figures will be saved to {FIG}/viz_*.png')

Figures will be saved to /Users/admin/Desktop/acoustic-delamination-detection/figures/viz_*.png


## 1. Dataset overview

Counts of pillars and clips per class, plus the holdout split.

In [2]:
def pillar_id(name):
    base = name.split('-')[0]
    return int(''.join(c for c in base if c.isdigit()))

good_files = sorted(glob.glob(os.path.join(ROOT, 'data', 'good', '*.wav')))
bad_files  = sorted(glob.glob(os.path.join(ROOT, 'data', 'bad',  '*.wav')))
good_pillars = sorted(set(pillar_id(os.path.basename(f)) for f in good_files))
bad_pillars  = sorted(set(pillar_id(os.path.basename(f)) for f in bad_files))

holdout_dirs = sorted(d for d in os.listdir(os.path.join(ROOT, 'data', 'holdout'))
                       if os.path.isdir(os.path.join(ROOT, 'data', 'holdout', d)))
holdout_clip_counts = {d: len(os.listdir(os.path.join(ROOT, 'data', 'holdout', d))) for d in holdout_dirs}

# durations
def durations(files):
    out = []
    for f in files[::5]:    # subsample for speed
        y, _ = librosa.load(f, sr=SR, mono=True); out.append(len(y)/SR*1000)
    return out

g_dur = durations(good_files); b_dur = durations(bad_files)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

# (a) per-class clip counts
ax = axes[0]
ax.bar(['Training\ngood', 'Training\nbad', 'Holdout\ngood', 'Holdout\nbad'],
       [len(good_files), len(bad_files),
        sum(c for d,c in holdout_clip_counts.items() if d.startswith('good')),
        sum(c for d,c in holdout_clip_counts.items() if d.startswith('bad'))],
       color=['#3498db', '#e74c3c', '#3498db', '#e74c3c'], edgecolor='black', linewidth=0.5)
ax.set_title('Clip counts')
ax.set_ylabel('# clips')
for i, v in enumerate([len(good_files), len(bad_files),
                       sum(c for d,c in holdout_clip_counts.items() if d.startswith('good')),
                       sum(c for d,c in holdout_clip_counts.items() if d.startswith('bad'))]):
    ax.text(i, v + 5, str(v), ha='center', fontsize=10)

# (b) per-class pillar counts
ax = axes[1]
ax.bar(['Training\ngood', 'Training\nbad', 'Holdout\ngood', 'Holdout\nbad'],
       [len(good_pillars), len(bad_pillars), 3, 3],
       color=['#3498db', '#e74c3c', '#3498db', '#e74c3c'], edgecolor='black', linewidth=0.5)
ax.set_title('Pillar counts')
ax.set_ylabel('# pillars')
for i, v in enumerate([len(good_pillars), len(bad_pillars), 3, 3]):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=10)

# (c) clip duration distribution
ax = axes[2]
ax.hist([g_dur, b_dur], bins=20, color=['#3498db', '#e74c3c'], label=['good', 'bad'], edgecolor='black', linewidth=0.3)
ax.set_xlabel('Clip duration (ms)')
ax.set_ylabel('# clips')
ax.set_title('Clip durations (subsampled)')
ax.legend()

plt.suptitle('Dataset overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_dataset_overview.png'), bbox_inches='tight')
plt.close()
print(f'Training: {len(good_pillars)} good + {len(bad_pillars)} bad pillars '
      f'({len(good_files)} + {len(bad_files)} clips)')
print(f'Holdout : 3 good + 3 bad pillars ({sum(holdout_clip_counts.values())} clips)')
print('Saved figures/viz_dataset_overview.png')

Training: 13 good + 18 bad pillars (245 + 263 clips)
Holdout : 3 good + 3 bad pillars (72 clips)
Saved figures/viz_dataset_overview.png


## 2. Raw waveforms — the physics

Side-by-side waveforms of representative good and bad knocks. Intact concrete rings; delaminated decays faster.


In [3]:
good_example = os.path.join(ROOT, 'data', 'good', 'goodConcrete1-03.wav')
bad_example  = os.path.join(ROOT, 'data', 'bad',  'badConcrete1-03.wav')

yg, _ = librosa.load(good_example, sr=SR, mono=True)
yb, _ = librosa.load(bad_example,  sr=SR, mono=True)
tg = np.arange(len(yg)) / SR * 1000
tb = np.arange(len(yb)) / SR * 1000

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
axes[0].plot(tg, yg, color='#3498db', linewidth=0.6); axes[0].set_title('GOOD — intact concrete', fontweight='bold')
axes[1].plot(tb, yb, color='#e74c3c', linewidth=0.6); axes[1].set_title('BAD — delaminated', fontweight='bold')
for ax in axes:
    ax.set_xlabel('Time (ms)'); ax.set_xlim(0, max(tg.max(), tb.max()))
axes[0].set_ylabel('Amplitude (normalized)')
plt.suptitle('Raw waveform comparison (one knock each)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_waveforms.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_waveforms.png')

Saved figures/viz_waveforms.png


## 3. Mel spectrograms — what the model sees

In [4]:
def mel_db(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, hop_length=64, n_mels=64, fmax=8000)
    return librosa.power_to_db(S, ref=np.max)

# Pad/crop to 200ms for fair comparison
def to_200ms(y):
    target = 3200
    if len(y) >= target: return y[:target]
    return np.pad(y, (0, target - len(y)))

Sg = mel_db(to_200ms(yg))
Sb = mel_db(to_200ms(yb))
diff = Sb - Sg

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ims = []
for ax, S, title, cmap in zip(axes, [Sg, Sb, diff],
                                ['GOOD mel-spec (dB)', 'BAD mel-spec (dB)', 'Difference (BAD − GOOD)'],
                                ['magma', 'magma', 'RdBu_r']):
    im = librosa.display.specshow(S, sr=SR, hop_length=64, x_axis='time', y_axis='mel',
                                   fmax=8000, ax=ax, cmap=cmap)
    ax.set_title(title, fontweight='bold')
    ims.append(im)
fig.colorbar(ims[0], ax=axes[0], format='%+2.0f dB')
fig.colorbar(ims[1], ax=axes[1], format='%+2.0f dB')
fig.colorbar(ims[2], ax=axes[2], format='%+2.0f dB')
plt.suptitle('Mel spectrograms — good vs bad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_mel_spectrograms.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_mel_spectrograms.png')


Saved figures/viz_mel_spectrograms.png


## 4. Feature space PCA

Reduces the 396-dim feature vector to 2D for visualization. Two views:
1. Training only — shows that good and bad classes separate cleanly with the v1 features.
2. Holdout overlay — shows that the unseen pillars sit on the correct side of the boundary.

In [5]:
Xtr, ytr, ptr, vtr, Xho, yho, pho, vho = pickle.load(
    open(os.path.join(ROOT, 'model', 'robust_v1_features_cache.pkl'), 'rb'))

sc = StandardScaler()
Xtr_s = sc.fit_transform(Xtr)
Xho_s = sc.transform(Xho)

pca = PCA(n_components=2)
Z_tr = pca.fit_transform(Xtr_s[vtr == 0])    # original training only
Z_ho = pca.transform(Xho_s)
y_tr_orig = ytr[vtr == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: training only
ax = axes[0]
ax.scatter(Z_tr[y_tr_orig==0, 0], Z_tr[y_tr_orig==0, 1], c='steelblue', s=14, alpha=0.5, label='train GOOD')
ax.scatter(Z_tr[y_tr_orig==1, 0], Z_tr[y_tr_orig==1, 1], c='crimson',   s=14, alpha=0.5, label='train BAD')
ax.set_title('Training feature space (PCA 2D)\nClasses separate with v1 features', fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend()

# Right: + holdout
ax = axes[1]
ax.scatter(Z_tr[y_tr_orig==0, 0], Z_tr[y_tr_orig==0, 1], c='steelblue', s=14, alpha=0.3, label='train GOOD')
ax.scatter(Z_tr[y_tr_orig==1, 0], Z_tr[y_tr_orig==1, 1], c='crimson',   s=14, alpha=0.3, label='train BAD')
ax.scatter(Z_ho[yho==0, 0], Z_ho[yho==0, 1], c='blue', marker='*', s=220,
            edgecolors='black', linewidths=1.2, label='HOLDOUT good (3 pillars)')
ax.scatter(Z_ho[yho==1, 0], Z_ho[yho==1, 1], c='red', marker='*', s=220,
            edgecolors='black', linewidths=1.2, label='HOLDOUT bad  (3 pillars)')
ax.set_title('With holdout overlay\nUnseen pillars land on the correct side', fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(loc='best', fontsize=9)

plt.suptitle('Feature space (v1 features + CMVN)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_feature_space.png'), bbox_inches='tight')
plt.close()
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')
print('Saved figures/viz_feature_space.png')

PCA explained variance: 38.8%
Saved figures/viz_feature_space.png


## 5. Final-model results

Refit the final ensemble (SVM + GBM + RF) and produce the four key result figures:
- Headline accuracy comparison
- Per-pillar P(bad) on holdout
- Per-clip predictions with abstention zone
- Confusion matrix

In [6]:
# Refit the ensemble (matches notebooks/Ensemble_Abstention_Model.ipynb)
models = {
    'SVM': SVC(kernel='rbf', C=5.0, gamma=0.001, class_weight='balanced', probability=True),
    'GBM': GradientBoostingClassifier(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=42),
    'RF':  RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42),
}
for name, clf in models.items():
    clf.fit(Xtr_s, ytr)

hp = {n: clf.predict_proba(Xho_s)[:, 1] for n, clf in models.items()}
ens = np.mean(np.stack(list(hp.values())), axis=0)

PILLARS = sorted(set(pho))
def clean_label(p):
    return ('good_' if p.startswith('good') else 'bad_') + p.split('_')[1][:-1]

def pillar_summary(probs, threshold=0.5):
    out = []
    for p in PILLARS:
        m = pho == p
        truth = int(yho[m][0])
        votes = (probs[m] >= threshold).astype(int)
        vote = int(np.bincount(votes, minlength=2).argmax())
        out.append((p, truth, vote, float(probs[m].mean()), int(m.sum())))
    return out

print('Ensemble refit done.')

Ensemble refit done.


In [7]:
# (5a) Headline accuracy bar chart
fig, ax = plt.subplots(figsize=(9, 5))
labels = ['Original\nSVM', 'v1 SVM', 'v1 GBM', 'v1 RF', 'v1\nEnsemble', 'Ensemble\n+ abstention*']
pillar_acc_vals = [50.0, 83.3, 100.0, 100.0, 83.3, 100.0]
clip_acc_vals   = [56.9, 91.7, 95.8,  95.8,  91.7, 100.0]
x = np.arange(len(labels)); w = 0.36
ax.bar(x - w/2, pillar_acc_vals, w, label='Per-pillar', color='#2c3e50')
ax.bar(x + w/2, clip_acc_vals, w, label='Per-clip', color='#e67e22')
ax.axhline(85, linestyle='--', linewidth=1, color='#27ae60', alpha=0.7)
ax.text(len(labels) - 0.5, 86, 'target 85%', color='#27ae60', fontsize=10, ha='right')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Holdout accuracy on 6 unseen pillars\n(* abstention: per-clip accuracy is on confident clips only; 4.2% of clips abstained)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 108)
ax.legend(loc='lower right')
for xi, (a, b) in enumerate(zip(pillar_acc_vals, clip_acc_vals)):
    ax.text(xi - w/2, a + 1.5, f'{a:.0f}', ha='center', fontsize=9)
    ax.text(xi + w/2, b + 1.5, f'{b:.0f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_headline_accuracy.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_headline_accuracy.png')


Saved figures/viz_headline_accuracy.png


In [8]:
# (5b) Per-pillar P(bad) bar chart (ensemble)
summary = pillar_summary(ens)
fig, ax = plt.subplots(figsize=(9, 5))
labels  = [clean_label(r[0]) for r in summary]
pbads   = [r[3] for r in summary]
truths  = [r[1] for r in summary]
colors  = ['#3498db' if t == 0 else '#e74c3c' for t in truths]
ax.bar(np.arange(len(labels)), pbads, color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0.5, linestyle='--', color='#555', alpha=0.7)
ax.text(len(labels) - 0.5, 0.52, 'decision threshold', color='#555', fontsize=9, ha='right')
ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('P(bad) — model confidence', fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Per-pillar holdout confidence (ensemble)\nblue = truth GOOD, red = truth BAD',
             fontsize=13, fontweight='bold')
for xi, p in enumerate(pbads):
    ax.text(xi, p + 0.02, f'{p*100:.0f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_per_pillar_confidence.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_per_pillar_confidence.png')

Saved figures/viz_per_pillar_confidence.png


In [9]:
# (5c) Per-clip predictions with abstention zone
ABST_LO, ABST_HI = 0.40, 0.60
fig, ax = plt.subplots(figsize=(11, 6))
x_pos = 0; xticks = []; xticklabels = []
for pi, p in enumerate(PILLARS):
    m = pho == p
    pr = ens[m]
    truth = int(yho[m][0])
    n = len(pr)
    xs = x_pos + np.arange(n) * 0.18
    pred = (pr >= 0.5).astype(int)
    correct = pred == truth
    abst = (pr >= ABST_LO) & (pr <= ABST_HI)
    colors = []
    for c, t, a in zip(correct, [truth]*n, abst):
        if a:                        colors.append('#f39c12')   # abstain (orange)
        elif c and t == 0:           colors.append('#27ae60')   # good correct
        elif c and t == 1:           colors.append('#c0392b')   # bad correct
        else:                        colors.append('#7f8c8d')   # wrong (grey)
    ax.scatter(xs, pr, c=colors, s=80, edgecolors='black', linewidths=0.5, alpha=0.9)
    xticks.append(x_pos + (n - 1) * 0.09)
    xticklabels.append(f'{clean_label(p)}\n(truth: {"GOOD" if truth==0 else "BAD"})\n{n} clips')
    if pi < len(PILLARS) - 1:
        ax.axvline(x_pos + n * 0.18, color='#bbb', linewidth=0.8, linestyle=':')
    x_pos += n * 0.18 + 0.4

# Abstention shaded band
ax.axhspan(ABST_LO, ABST_HI, color='#f39c12', alpha=0.10, zorder=0)
ax.axhline(0.5, linestyle='--', color='#444', linewidth=1)
ax.text(ax.get_xlim()[1], (ABST_LO+ABST_HI)/2, ' abstention zone', color='#d35400', fontsize=10, va='center')
ax.set_ylabel('P(bad) per clip', fontsize=12)
ax.set_xticks(xticks); ax.set_xticklabels(xticklabels, fontsize=10)
ax.set_ylim(-0.03, 1.05)
ax.set_title('Per-clip holdout predictions (ensemble)\nGreen=GOOD correct, Red=BAD correct, Orange=abstained, Grey=misclassified',
             fontsize=13, fontweight='bold')
legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#27ae60', markersize=10,
           markeredgecolor='black', label='GOOD correctly classified'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#c0392b', markersize=10,
           markeredgecolor='black', label='BAD correctly classified'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#f39c12', markersize=10,
           markeredgecolor='black', label='Abstain (knock again)'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#7f8c8d', markersize=10,
           markeredgecolor='black', label='Misclassified (confidently wrong)'),
]
ax.legend(handles=legend_handles, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_per_clip_with_abstention.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_per_clip_with_abstention.png')


Saved figures/viz_per_clip_with_abstention.png


In [10]:
# (5d) Confusion matrix on holdout (per-pillar, ensemble + abstention)
true_p = []; pred_p = []
for p in PILLARS:
    m = pho == p; pr = ens[m]
    confident = (pr < ABST_LO) | (pr > ABST_HI)
    if not confident.any(): continue
    truth = int(yho[m][0])
    vote = int(np.bincount((pr[confident] >= 0.5).astype(int), minlength=2).argmax())
    true_p.append(truth); pred_p.append(vote)
cm = confusion_matrix(true_p, pred_p, labels=[0, 1])

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap='Blues', vmin=0, vmax=cm.max())
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=22,
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['predicted GOOD', 'predicted BAD'], fontsize=11)
ax.set_yticklabels(['truth GOOD', 'truth BAD'], fontsize=11)
ax.set_title('Holdout confusion matrix\n(ensemble + abstention, per-pillar)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_confusion_matrix.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_confusion_matrix.png')

Saved figures/viz_confusion_matrix.png


## 6. Pipeline diagram

Schematic of the deployment pipeline (no audio data — pure layout).

In [11]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.set_xlim(0, 14); ax.set_ylim(0, 6); ax.axis('off')

def box(x, y, w, h, txt, fc='#ecf0f1', fontsize=11, fontweight='normal'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.06',
                                 facecolor=fc, edgecolor='#2c3e50', linewidth=1.2))
    ax.text(x + w/2, y + h/2, txt, ha='center', va='center',
            fontsize=fontsize, fontweight=fontweight)

def arrow(x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Stages
box(0.2, 4.5, 1.8, 1.0, 'knock\n(200ms)', fc='#fff5b1', fontweight='bold')
box(2.4, 4.5, 1.9, 1.0, 'features\n(396-dim)', fc='#dff9fb')
box(4.7, 4.5, 1.9, 1.0, 'CMVN\n(channel-invariant)', fc='#dff9fb', fontsize=10)

# 3 models
box(7.4, 5.2, 1.6, 0.8, 'SVM',  fc='#fadbd8')
box(7.4, 4.2, 1.6, 0.8, 'GBM',  fc='#fadbd8', fontweight='bold')
box(7.4, 3.2, 1.6, 0.8, 'RF',   fc='#fadbd8')

# Average
box(9.5, 4.2, 1.7, 0.8, 'mean P(bad)', fc='#d5f5e3')

# Decision
box(11.6, 5.2, 2.2, 0.8, 'P > 0.6  →  BAD',  fc='#e74c3c', fontsize=10)
box(11.6, 4.2, 2.2, 0.8, 'P ∈ [0.4, 0.6]\nABSTAIN',     fc='#f39c12', fontsize=9, fontweight='bold')
box(11.6, 3.0, 2.2, 0.8, 'P < 0.4  →  GOOD', fc='#27ae60', fontsize=10)

# Arrows
arrow(2.0, 5.0, 2.4, 5.0)
arrow(4.3, 5.0, 4.7, 5.0)
arrow(6.6, 5.0, 7.4, 5.6); arrow(6.6, 5.0, 7.4, 4.6); arrow(6.6, 5.0, 7.4, 3.6)
arrow(9.0, 5.6, 9.5, 4.8); arrow(9.0, 4.6, 9.5, 4.6); arrow(9.0, 3.6, 9.5, 4.4)
arrow(11.2, 4.6, 11.6, 5.6); arrow(11.2, 4.6, 11.6, 4.6); arrow(11.2, 4.6, 11.6, 3.4)

# Caption text
ax.text(7, 1.5, 'In a 5-knock session, the pillar verdict is the majority vote of confident clips.\n'
                'If too few clips are confident, the app says "knock again".',
        ha='center', fontsize=11, style='italic', color='#555')
ax.text(7, 6.2, 'Concrete delamination classifier — deployment pipeline',
        ha='center', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG, 'viz_pipeline.png'), bbox_inches='tight')
plt.close()
print('Saved figures/viz_pipeline.png')

Saved figures/viz_pipeline.png


## All figures saved

```
figures/viz_dataset_overview.png
figures/viz_waveforms.png
figures/viz_mel_spectrograms.png
figures/viz_feature_space.png
figures/viz_headline_accuracy.png
figures/viz_per_pillar_confidence.png
figures/viz_per_clip_with_abstention.png
figures/viz_confusion_matrix.png
figures/viz_pipeline.png
```

Suggested order for slides:
1. `viz_dataset_overview` — what we have
2. `viz_waveforms` + `viz_mel_spectrograms` — the physics
3. `viz_pipeline` — the system
4. `viz_feature_space` — features separate the classes
5. `viz_headline_accuracy` — money slide
6. `viz_per_clip_with_abstention` — show how the abstention works
7. `viz_confusion_matrix` — final result